# Q1b: Forecast avg time spent in the mall by a vehicle entering on a particular day, for next 7 days

**INSTRUCTIONS TO USER:**
1. Use the same `parkingLot.csv`.
2. Run the cells below after adjusting the data path.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
# 1. Load and Clean Data
df = pd.read_csv('parkingLot.csv', dtype={'camera_id': str})
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Filter closed hours and handle missing values
df = df.dropna()
df = df[df['timestamp'].dt.hour >= 5]

# Ensure 'vehicle_no' OCR is consistent
df['vehicle_no'] = df['vehicle_no'].str.replace('O', '0')

# 2. Calculate time spent
# Sort by timestamp (required for merge_asof)
df = df.sort_values('timestamp')

entry_df = df[df['camera_id'] == '001'].copy()
exit_df = df[df['camera_id'] == '002'].copy()

# Keep a separate column for exit time before merge_asof overwrites the right timestamp
exit_df['exit_timestamp'] = exit_df['timestamp']

# Use merge_asof to find the closest subsequent exit for each entry
merged_df = pd.merge_asof(
    entry_df,
    exit_df[['vehicle_no', 'timestamp', 'exit_timestamp']],
    on='timestamp',
    by='vehicle_no',
    direction='forward'
)

# Rename the entry timestamp for clarity
merged_df = merged_df.rename(columns={'timestamp': 'entry_timestamp'})

# Drop entries that never had a subsequent exit (e.g., end of the dataset)
merged_df = merged_df.dropna(subset=['exit_timestamp'])

# Calculate time spent in hours
merged_df['time_spent_hours'] = (merged_df['exit_timestamp'] - merged_df['entry_timestamp']).dt.total_seconds() / 3600

# 3. Aggregate average time spent per day
# Using resample('D') ensures we have a continuous time series index
avg_time_daily = merged_df.set_index('entry_timestamp').resample('D')['time_spent_hours'].mean()

# Forward fill in case there are days with no vehicles (keeps TS intact)
avg_time_daily = avg_time_daily.ffill()

# 4. Train/Test Split
train = avg_time_daily[:-7]
test = avg_time_daily[-7:]


In [7]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error
import numpy as np

# 5. Modeling (Using ARIMA to capture weekly lags)
model = ARIMA(train, order=(7,1,1))
model_fit = model.fit()

# 6. Forecasting
forecast = model_fit.forecast(steps=7)

# 7. Evaluation
# MAPE implementation
mape = mean_absolute_percentage_error(test, forecast)
print(f"MAPE: {mape:.4f}")

# MASE implementation
def mase(y_true, y_pred, y_train):
    """
    Calculates the Mean Absolute Scaled Error (MASE).
    Denominator is the mean absolute error of the naive one-step in-sample forecast.
    """
    n = len(y_train)
    d = np.abs(np.diff(y_train)).sum() / (n - 1)
    errors = np.abs(y_true - y_pred)
    return errors.mean() / d

print(f"MASE: {mase(test, forecast, train):.4f}")


MAPE: 0.0121
MASE: 0.0812


c:\Users\multi\Data-DaVinci\Week 4\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
